# Retrieval Agents — 검색하는 에이전트 (Agentic RAG, 데스크탑 Ollama판)

코스 섹션 [Building Agentic RAG Systems](https://huggingface.co/learn/agents-course/unit2/smolagents/retrieval_agents)를 내 데스크탑 Ollama(`qwen3.5:9b`)로 해 봐요.

**한 문장 요약: RAG = 찾아서(검색) → 붙여서 → 답하기. Agentic RAG = 그 "찾기"를 에이전트가 직접 몇 번이고, 검색어도 바꿔 가며 한다**

| 단계 | 하는 일 | 모델 필요? |
|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | 1단계부터 |
| 2 | 웹 검색으로 RAG (원문 예제 ①) → 에이전트가 **검색을 몇 번, 무슨 말로** 하는지 보기 | ✅ (+ 인터넷) |
| 3 | 내 지식 창고 만들기: 문서 → **조각(chunk)** 으로 자르기 | ❌ |
| 4 | 모델 없이 **BM25 검색만** 해 보기 → 원문 예제의 함정 2개 발견 | ❌ |
| 5 | 검색기를 **도구**로 감싸 에이전트에게 주기 (원문 예제 ②) + 이전 대화 기억하기 | ✅ |
| 6 | 도구 2개(지식 창고 + 웹) → 에이전트가 **알아서 골라 쓰는지** | ✅ (+ 인터넷) |
| 7 | (선택) **뜻으로** 찾는 검색 = 임베딩 | 임베딩 모델 (따로 받아야 함) |
| 8 | 원문의 고급 전략 7가지 ↔ 이 노트북에서 본 것 (읽기만) | — |

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 이번엔 **새 패키지 3개**(`langchain-community`, `langchain-text-splitters`, `rank_bm25`)가 필요해요 → `git pull` 뒤 `pip install -r requirements.txt` 한 번.
❌ 표시 칸은 모델을 안 불러서 **바로** 끝나요.

## 0단계: 준비 확인

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "langchain_community", "langchain_text_splitters", "rank_bm25"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)
print("(7단계 선택용) nomic-embed-text 있음?", any(m.startswith("nomic-embed-text") for m in models))

✅ **확인할 것**: `smolagents 버전: 1.26.0`, `qwen3.5:9b`와 패키지 5개 전부 `True`. 마지막 줄(`nomic-embed-text`)은 `False`여도 괜찮아요 (7단계만 건너뜀)
- `langchain_community 설치됨? False` → 아직 새 패키지를 안 깔았음: Jupyter를 끄고 `pip install -r requirements.txt`
- `ModuleNotFoundError` → venv를 켜지 않고 Jupyter를 열었음
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결

지난 노트북들과 **같은 설정**이에요.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말

## 2단계: 웹 검색으로 RAG (원문 예제 ①)

RAG의 세 글자를 이 예제에 대 보면:

| RAG | 뜻 | 여기서는 |
|---|---|---|
| **R**etrieval (찾기) | 질문과 관련된 자료를 찾음 | `web_search` 도구로 DuckDuckGo 검색 |
| **A**ugmented (붙이기) | 찾은 자료를 모델이 보는 글에 붙임 | 검색 결과가 **관찰(Observation)** 로 기억에 들어감 |
| **G**eneration (답 쓰기) | 자료를 보고 답을 씀 | `final_answer(...)` |

**보통 RAG**는 "질문 그대로 **한 번** 검색 → 답"으로 순서가 정해져 있어요.
**Agentic RAG**는 에이전트가 검색어를 직접 만들고, 결과가 모자라면 **다시** 찾아요. 이 칸에서 그걸 눈으로 봐요.

원문 코드에서 `InferenceClientModel()` → `model`만 바꾸고, 너무 오래 맴돌지 않게 `max_steps=6`을 넣었어요 (원문 기본값은 20).
인터넷이 필요하고, 몇 분 걸릴 수 있어요.

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool

search_tool = DuckDuckGoSearchTool()

agent = CodeAgent(
    model=model,
    tools=[search_tool],
    max_steps=6,  # 원문엔 없음: 최대 6바퀴까지만
)

response = agent.run(
    "Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering."
)
print(response)

✅ **확인할 것** (화면에 찍힌 Step들을 위에서부터 훑어보기)
- `web_search(...)`를 **몇 번** 불렀나? 한 번에 끝났나, 꾸미기·공연·음식을 **나눠서** 찾았나?
- 검색어가 내 질문과 **똑같나, 바뀌었나?** (예: "luxury superhero party decorations" 처럼 짧게 다듬었으면 → 8단계의 Query Reformulation/Decomposition)
- 마지막 답에 검색 결과 내용이 섞여 있나, 모델이 원래 아는 상식인가?

⚠️ 원문 설명 4번 "Stores for Future Reference(나중을 위해 저장)"는 좀 과장이에요. 에이전트는 **이번 실행 안에서만** 기억해요 (`agent.memory`). 다음 `agent.run(...)`을 부르면 기본으로 기억을 **지워요**(`reset=True`). 이어서 기억하게 하는 법은 5-B단계에서 봐요.

## 3단계: 내 지식 창고 만들기 — 문서 → 조각(chunk)

웹 말고 **내 자료**에서 찾게 하려면 먼저 자료를 검색하기 좋게 준비해요. 원문 예제 ②의 앞부분이에요.

| 단어 | 뜻 | 비유 |
|---|---|---|
| `Document` | 글 한 덩어리(`page_content`) + 꼬리표(`metadata`, 예: 출처) | 도서관 책 + 책 등에 붙은 라벨 |
| text splitter | 긴 글을 작은 **조각(chunk)** 으로 자름 | 두꺼운 책을 쪽지로 잘라 카드 상자에 넣기 |
| 왜 자르나? | 긴 글 통째로는 ① 검색이 둔해지고 ② 모델 입력창(`num_ctx`)을 금방 채움 | 질문에 딱 맞는 쪽지 몇 장만 꺼내 보기 |

> 원문은 `from langchain_community.docstore.document import Document`로 가져오는데, 지금 버전에선 그게 `langchain_core`의 `Document`를 다시 내보내는 것뿐이에요 (같은 물건). 여기선 원래 자리인 `langchain_core.documents`에서 가져와요.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 파티 아이디어 지식 창고 (원문 그대로, 가짜 데이터 5개)
party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"},
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # 조각 하나의 최대 글자 수
    chunk_overlap=50,        # 이웃 조각끼리 겹치게 둘 글자 수
    add_start_index=True,    # 원래 글의 몇 번째 글자에서 시작했는지 metadata에 적기
    strip_whitespace=True,   # 조각 앞뒤 공백 지우기
    separators=["\n\n", "\n", ".", " ", ""],  # 자를 자리 우선순위: 문단 → 줄 → 문장 → 단어 → 글자
)
docs_processed = text_splitter.split_documents(source_docs)

print("원래 문서 수:", len(source_docs), "→ 조각 수:", len(docs_processed))
for d in docs_processed:
    print(f"- [{d.metadata['source']}] (start_index={d.metadata['start_index']}, {len(d.page_content)}자) {d.page_content[:50]}...")

✅ **확인할 것**: `원래 문서 수: 5 → 조각 수: 5`
- 문서가 전부 **500자보다 짧아서 하나도 안 잘렸어요.** 원문 예제에선 splitter가 사실상 아무 일도 안 해요. 진짜 쓰임은 **긴 글**일 때라서, 바로 아래 3-B에서 긴 글로 잘라 봐요
- `metadata`에 `source`(내가 붙인 출처)와 `start_index`(splitter가 붙인 시작 위치)가 같이 있어요 → 나중에 "이 답은 어느 문서에서 왔나"를 밝힐 때 써요

### 3-B: 긴 글을 작게 잘라 보기 — `chunk_overlap`이 뭔지 눈으로

Wayne Manor 안내문(가짜, 약 270자)을 `chunk_size=100`, `chunk_overlap=30`으로 잘라요. 겹침이 잘 보이게 **단어 단위**(`separators=[" "]`)로만 자르게 했어요.

In [ ]:
manor_guide = (
    "Wayne Manor has a grand ballroom on the first floor. The ballroom fits about 200 guests. "
    "The garden behind the manor is good for an outdoor cocktail hour. Guests should not enter the Batcave under the manor. "
    "The kitchen can serve hot meals for up to 150 people at once."
)

small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=30,
    add_start_index=True,
    strip_whitespace=True,
    separators=[" "],  # 겹침을 보기 쉽게 단어 단위로만
)
chunks = small_splitter.split_documents([Document(page_content=manor_guide, metadata={"source": "Manor Guide"})])

print("전체 길이:", len(manor_guide), "자 → 조각", len(chunks), "개\n")
for i, c in enumerate(chunks):
    print(f"조각 {i} (start_index={c.metadata['start_index']}): {c.page_content!r}\n")

✅ **확인할 것**: 조각 4개. **앞 조각의 끝 몇 단어가 다음 조각 앞에 또 나와요** (예: 조각 0 끝 `about 200 guests. The garden` → 조각 1 시작 `about 200 guests. The garden`)
- 이게 `chunk_overlap`: 자르는 자리에 걸친 문장이 반으로 쪼개져 뜻을 잃지 않게, **경계 부분을 양쪽 조각에 다 넣어 두는 것**. 책을 복사할 때 쪽 끝 몇 줄을 다음 장에도 겹쳐 찍는 것과 같아요
- 참고: 원문 설정(`separators`에 `"."` 포함)으로 자르면 문장 단위로 잘려서 **겹침이 잘 안 생겨요.** 겹침은 "앞 조각의 끝 덩어리를 통째로" 다시 붙이는데, 문장 하나가 30자보다 길면 붙일 수 없거든요

## 4단계: 모델 없이 BM25 검색만 해 보기 — 원문 예제의 함정 2개

**BM25** = 아주 오래되고 튼튼한 **단어 맞추기(키워드)** 검색 방법이에요. 점수 계산의 감은 이래요:
- 질문의 단어가 그 조각에 **많이** 나올수록 ↑
- 그 단어가 **드문 단어**일수록 ↑ (모든 조각에 다 있는 `the` 같은 단어는 별 의미 없음)
- 조각이 너무 길면 조금 ↓

⚠️ 원문 도구 설명엔 "Uses **semantic search**(뜻으로 찾기)"라고 돼 있지만, **BM25는 뜻이 아니라 단어 글자가 같은지만** 봐요. 진짜 "뜻으로 찾기"는 7단계(임베딩)예요.

원문처럼 `k=5`(상위 5개 돌려주기)로 만들고 아무거나 검색해 봐요.

In [ ]:
from langchain_community.retrievers import BM25Retriever

retriever_k5 = BM25Retriever.from_documents(docs_processed, k=5)  # 원문 설정

for q in ["catering", "music", "spaceship"]:
    found = retriever_k5.invoke(q)
    print(f"{q!r:12} → {len(found)}개: {[d.metadata['source'] for d in found]}")

✅ **확인할 것**: 무슨 말로 검색해도(`spaceship`처럼 상관없는 말도) **5개 전부** 나와요

**함정 ①: 조각이 5개인데 `k=5` → 검색이 아니라 "전부 보여 주기"**예요. 원문 예제는 모델이 그냥 지식 창고 전체를 받아 보는 셈이에요. 그래도 잘 되는 건 자료가 작아서일 뿐. 진짜 검색이 일어나게 **`k=2`** 로 줄여 보고, 점수도 직접 봐요.

In [ ]:
retriever_k2 = BM25Retriever.from_documents(docs_processed, k=2)

def show_scores(retriever, query):
    words = retriever.preprocess_func(query)          # 질문을 단어로 쪼갠 결과
    scores = retriever.vectorizer.get_scores(words)   # 조각마다 BM25 점수
    print(f"\n질문 {query!r} → 단어 {words}")
    for doc, s in zip(docs_processed, scores):
        print(f"   {s:5.2f}  [{doc.metadata['source']}] {doc.page_content[:45]}...")
    print("   ▶ 돌려준 2개:", [d.metadata["source"] for d in retriever.invoke(query)])

show_scores(retriever_k2, "catering")
show_scores(retriever_k2, "DJ music")

print("\n----- BM25가 조각을 단어로 쪼갠 모양 (Catering 조각) -----")
print(retriever_k2.preprocess_func(docs_processed[2].page_content))

✅ **확인할 것**
- `DJ music` → Entertainment(DJ) 조각만 점수가 높고 1등 ✅ (단어가 글자 그대로 있으니까)
- `catering` → **점수가 전부 0.00**, 그래서 Catering 조각이 아니라 아무거나 2개가 나와요 ❌

**함정 ②: 단어 쪼개기가 너무 단순해요.** 맨 아래 줄을 보면 Catering 조각이 `'For'`, **`'catering,'`**(쉼표 붙음), `'superheroes,'` … 로 쪼개져 있어요.
기본 설정은 **띄어쓰기로만 자르고, 대소문자·쉼표를 그대로** 둬서 `catering` ≠ `catering,`, `for` ≠ `For`가 돼요.

→ 고치기: 쪼개기 함수(`preprocess_func`)를 **"소문자로 바꾸고, 영어·숫자만 단어로"** 로 바꿔요.

In [ ]:
import re


def simple_tokenize(text: str) -> list[str]:
    # 소문자로 바꾸고, 알파벳·숫자 덩어리만 단어로 뽑기 → 쉼표·따옴표·대소문자 문제 해결
    return re.findall(r"[a-z0-9]+", text.lower())


print(simple_tokenize("For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie'"))

retriever_fixed = BM25Retriever.from_documents(docs_processed, k=2, preprocess_func=simple_tokenize)

show_scores(retriever_fixed, "catering")
show_scores(retriever_fixed, "food")
show_scores(retriever_fixed, "decoration")

✅ **확인할 것**
- 이제 `catering` → **Catering Ideas가 1등** ✅
- 그런데 `food` → 다시 **전부 0.00** ❌. 지식 창고엔 `food`라는 단어가 없거든요 (`dishes`, `catering`, `smoothie`, `steak`는 있는데)
- `decoration`도 **전부 0.00** ❌. 본문엔 `Decorate`(동사)만 있고, `Decoration`은 출처 꼬리표(`metadata`)에만 있어요. BM25는 ① **본문(`page_content`)만** 보고 ② `decorate`와 `decoration`을 **다른 단어**로 봐요

이게 **키워드 검색의 한계**: 뜻이 같아도 **글자가 다르면 못 찾아요.** 해결 방법은 두 갈래예요.
1. **검색어를 바꿔서 다시 찾기** (Query Reformulation) → 에이전트가 "food로 안 나오네? catering으로 찾자"를 **스스로** 하면 = Agentic RAG. 5단계에서 봐요
2. **뜻으로 찾는 검색**(임베딩) 쓰기 → 7단계(선택)

## 5단계: 검색기를 도구로 감싸 에이전트에게 주기 (원문 예제 ②)

5번 Tools 섹션의 **`Tool` 클래스 상속** 방식 그대로예요. 설명서 4칸(`name`·`description`·`inputs`·`output_type`) + 실제 일(`forward`).
원문에서 바꾼 것 3가지 (주석에 `★`):
1. `k=5` → `k=2` (진짜로 고르게)
2. 단어 쪼개기를 `simple_tokenize`로 (함정 ② 해결)
3. `forward` 안에 `print`를 넣어 **에이전트가 실제로 어떤 검색어를 썼는지** 보이게

In [ ]:
from smolagents import CodeAgent, Tool  # CodeAgent도 여기서 다시 가져옴 (2단계를 건너뛰어도 되게)


class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Uses semantic search to retrieve relevant party planning ideas for Alfred's superhero-themed party at Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be a query related to party planning or superhero themes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs,
            k=2,                             # ★ 원문은 5 (= 전부)
            preprocess_func=simple_tokenize,  # ★ 원문엔 없음 (함정 ② 해결)
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"
        print(f"   🔎 지식 창고 검색어: {query!r}")  # ★ 원문엔 없음 (관찰용)

        docs = self.retriever.invoke(query)
        return "\nRetrieved ideas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )


party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

# 모델 없이 도구만 먼저 불러 보기
print(party_planning_retriever("catering"))

✅ **확인할 것**: `🔎 지식 창고 검색어: 'catering'` 다음에 Idea 0이 **Hulk's Green Smoothie** 조각
- 도구가 돌려주는 건 그냥 **글(string)** 이에요. 에이전트는 이 글을 관찰(Observation)로 받아요 = RAG의 "붙이기(Augmented)"

이제 에이전트에게 줘요. 원문 질문 그대로예요.

In [ ]:
agent = CodeAgent(tools=[party_planning_retriever], model=model, max_steps=6)

response = agent.run(
    "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options."
)
print("\n===== 최종 답 =====")
print(response)

✅ **확인할 것**: `🔎 지식 창고 검색어:` 줄을 세어 보기
- 한 번에 질문 통째로 검색했나, **entertainment / catering / decoration으로 나눠서** 3번 찾았나? 나눴다면 그게 8단계의 **Query Decomposition(질문 쪼개기)** 이에요. 우리가 시킨 게 아니라 모델이 정한 거예요
- `k=2`라 한 번 검색에 2개만 와요. 그래도 최종 답에 5개 아이디어(가면무도회·DJ·음식·로고·VR)가 많이 들어갔다면, 여러 번 나눠 찾은 덕분이에요
- 원문 `k=5`였으면 첫 검색 한 번에 전부 받아서 나눌 필요가 없었을 거예요
- 🔎 아래 결과가 엉뚱하면(예: `entertainment`로 찾았는데 DJ가 아니라 VR·로고가 옴): 4단계의 `decoration`처럼 **본문에 없는 단어**라 점수가 전부 0이 된 거예요. 클라우드에서 가짜 모델로 `entertainment`·`catering`·`decoration` 세 번 찾게 해 봤더니 `catering`만 제대로 나왔어요. 모델이 결과를 보고 **검색어를 바꿔 다시 찾으면**(`DJ`, `music`, `decorate`…) 그게 Agentic RAG가 잘 돌아가는 모습이에요

### 5-B: 이전 대화 기억하기 — `reset=False`

원문의 "Maintain conversation context in memory(대화 맥락을 기억)"를 직접 봐요. 같은 `agent`에게 `reset=False`로 **이어서** 물어요.

In [ ]:
follow_up = agent.run(
    "Which ONE of the ideas you just found would be the cheapest to do? Answer in one sentence.",
    reset=False,  # ★ 기억을 지우지 않고 이어서
)
print("\n===== 이어서 물은 답 =====")
print(follow_up)
print("\n지금 기억에 쌓인 단계 수:", len(agent.memory.steps))

✅ **확인할 것**
- 지식 창고를 **다시 검색하지 않고**(`🔎` 줄이 없거나 적고) 바로 "아까 찾은 것 중 이것"이라고 답하면 = 앞의 검색 결과를 **기억**에서 꺼내 쓴 것. 원문의 "Memory systems help … avoid repetitive retrievals(같은 검색 반복 피하기)"가 이거예요
- `reset=False`를 빼고(기본 `reset=True`) 다시 해 보면, "you just found"가 뭔지 몰라서 **다시 검색하거나 엉뚱하게** 답할 거예요 (해 보고 싶으면 한 줄만 고쳐서)
- 이 기억은 **노트북 커널이 켜져 있는 동안만**이에요. 파일에 저장되진 않아요 (2단계 ⚠️와 같은 얘기)

## 6단계: 도구 2개 → 에이전트가 알아서 골라 쓰나 (Source Integration)

지식 창고(파티 아이디어) + 웹 검색을 **둘 다** 줘요. 질문엔 지식 창고에 있는 것(꾸미기·음식)과 **없는 것**(요즘 슈퍼히어로 영화)이 섞여 있어요.
**어느 도구를 쓰라고는 안 알려 줘요.** 모델이 도구 설명서만 보고 고르는지 봐요.
인터넷이 필요해요.

In [ ]:
from smolagents import DuckDuckGoSearchTool

agent = CodeAgent(
    tools=[party_planning_retriever, DuckDuckGoSearchTool()],
    model=model,
    max_steps=6,
)

response = agent.run(
    "Plan the decorations and catering for a luxury superhero party at Wayne Manor, "
    "and also recommend one recent superhero movie we could screen for the guests."
)
print("\n===== 최종 답 =====")
print(response)

✅ **확인할 것**
- 꾸미기·음식은 `party_planning_retriever`(🔎 줄)로, 영화는 `web_search`로 찾았나? → 원문 "The agent **should select between available tools based on the query type**(질문 종류에 맞는 도구 고르기)"
- 둘 다 안 쓰고 모델 상식으로 영화를 답했다면? → 도구를 고를지 말지도 모델 판단. 영화가 진짜 최신인지는 **모름**(모델 지식엔 날짜 한계가 있음)
- 이게 1번 노트에 적은 **fallback(플랜 B)** 의 모양이에요: 지식 창고에 없으면 웹으로

## 7단계 (선택): 뜻으로 찾는 검색 = 임베딩

4단계에서 `food`로는 못 찾았죠. **임베딩(embedding)** 은 글을 **뜻을 담은 숫자 목록(벡터)** 으로 바꿔요. 뜻이 비슷한 글은 숫자도 비슷하게 나와서, 글자가 달라도 찾아요. 이렇게 만든 벡터를 쌓아 둔 창고가 **벡터 데이터베이스(vector store)** 예요.

| | BM25 (4~5단계) | 임베딩 검색 (이 단계) |
|---|---|---|
| 무엇을 비교 | 단어 **글자**가 같은가 | **뜻**이 가까운가 (벡터 사이 각도, 코사인 유사도) |
| `food` → Catering 조각 | ❌ 못 찾음 | ✅ 찾을 가능성 높음 |
| 필요한 것 | 없음 (계산만) | **임베딩 모델** |

**준비 (한 번만, 약 270MB 다운로드)**: 이 단계를 하고 싶을 때만 PowerShell에서
```
ollama pull nomic-embed-text
```
받은 뒤 **0단계를 다시 실행**하고 이 칸을 실행해요. 안 받았으면 이 칸은 안내만 찍고 넘어가요. 새 파이썬 패키지는 필요 없어요 (Ollama에 직접 요청).

In [ ]:
import math

EMBED_MODEL = "nomic-embed-text"


def embed(texts: list[str]) -> list[list[float]]:
    # Ollama에게 "이 글들을 벡터로 바꿔 줘" 요청
    body = json.dumps({"model": EMBED_MODEL, "input": texts}).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/embed", data=body, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read())["embeddings"]


def cosine(a: list[float], b: list[float]) -> float:
    # 두 벡터가 같은 방향이면 1에 가깝고, 상관없으면 0 근처
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


if not any(m.startswith(EMBED_MODEL) for m in models):
    print(f"'{EMBED_MODEL}'이 없어서 건너뛰어요. 하려면 PowerShell에서 ollama pull {EMBED_MODEL} → 0단계 다시 실행")
else:
    chunk_vectors = embed([d.page_content for d in docs_processed])
    print("조각 하나의 벡터 길이(숫자 개수):", len(chunk_vectors[0]))

    for q in ["food", "something to eat", "dance music", "food for guests"]:
        q_vec = embed([q])[0]
        ranked = sorted(
            zip(docs_processed, (cosine(q_vec, v) for v in chunk_vectors)),
            key=lambda pair: pair[1],
            reverse=True,
        )
        print(f"\n질문 {q!r}")
        for doc, s in ranked[:2]:
            print(f"   {s:.3f}  [{doc.metadata['source']}] {doc.page_content[:45]}...")

✅ **확인할 것** (받았을 때만)
- `food`, `something to eat` → **Catering Ideas가 1~2등**이면 성공: 조각에 `food`라는 글자가 없는데도 찾았어요 = **뜻으로 찾기(semantic search)**
- 점수는 0~1 사이 숫자. 크기 자체보다 **순서**를 보면 돼요
- 실전 RAG는 이 방식(벡터 DB)이나, BM25 + 임베딩을 **섞어** 써요. 원문이 BM25를 쓴 건 설치·설명이 쉬워서일 거예요 (미확인)

## 8단계 (읽기만): 원문의 고급 전략 7가지 ↔ 이 노트북에서 본 것

| # | 전략 | 한 줄 | 이 노트북에서 |
|---|---|---|---|
| 1 | Query Reformulation (검색어 다듬기) | 사용자 말 그대로 말고, 문서에 잘 맞는 검색어로 바꿔 찾기 | 2·5단계 검색어가 질문과 달랐나 / 4단계 `food` → `catering` |
| 2 | Query Decomposition (질문 쪼개기) | 여러 가지를 묻는 질문을 검색 여러 개로 나누기 | 5단계 entertainment / catering / decoration 따로 |
| 3 | Query Expansion (검색어 넓히기) | 같은 뜻을 **여러 말로** 바꿔 전부 검색 | `food`, `meal`, `dishes`로 다 찾아 보기 |
| 4 | Reranking (다시 줄 세우기) | 일단 넉넉히 찾은 뒤, 더 똑똑한 모델(**Cross-Encoder**)로 순위를 다시 매김 | 안 해 봄 (모델이 하나 더 필요) |
| 5 | Multi-Step Retrieval (여러 번 찾기) | 첫 검색 결과를 보고 다음 검색어를 정함 | 2단계에서 여러 번 검색했다면 |
| 6 | Source Integration (출처 섞기) | 웹 + 내 문서처럼 여러 곳에서 모아 합치기 | 6단계 |
| 7 | Result Validation (결과 확인) | 찾은 내용이 맞는지·관련 있는지 따져 본 뒤 답에 넣기 | 4단계 `spaceship`에도 5개가 왔음 → 확인 없이 쓰면 엉뚱한 자료가 섞임 |

**Cross-Encoder** 감 잡기: 7단계 임베딩은 질문과 조각을 **따로** 숫자로 바꿔 비교(빠름, 덜 정확) / Cross-Encoder는 질문+조각을 **한 번에 같이** 읽고 "얼마나 맞나" 점수를 냄(느림, 더 정확) → 그래서 "넉넉히 찾기(빠른 방법) → 상위 몇 개만 다시 줄 세우기(정확한 방법)" 순서로 써요.

**끝!** 결과(특히 2·5·6단계의 `🔎` 검색어와 `web_search` 횟수, 7단계를 했다면 점수)를 복사해서 AI에게 붙여 주면 같이 분석해요.
Jupyter 끄기: `Esc` → `Ctrl + C` 두 번.